# `create_train_aug.py` — Build Augmented Training Set

## Purpose
Merges the original (cleaned) training data with **LLM-generated synthetic review CSVs** to create a larger, more balanced training dataset.

## Outputs
| File | Description |
|------|-------------|
| `data/splits/train_augmented.csv` | Combined training set (original + synthetic) |
| `augmentation_impact.md` | Before/after class distribution report |

## Why augment?
The original dataset is severely class-imbalanced. For rare classes like `price-negative` (only ~17 samples!), the model struggles to learn meaningful decision boundaries. Adding LLM-generated synthetic reviews targeting those rare classes improves per-class F1 without introducing real label noise.

## Deduplication
- **Internal deduplication**: LLMs often repeat nearly-identical sentences. We drop exact-text duplicates within the synthetic data.
- **Cross-set deduplication**: Synthetic rows that appear verbatim in the original training set are removed to prevent data leakage.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import logging

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
log = logging.getLogger(__name__)

## Configuration

All paths and aspect names are defined here. These must match the values in `configs/config.yaml`.

In [ ]:
import os
ML_RESEARCH = os.path.dirname(os.path.abspath(''))

# Paths (relative to ml-research root — run this script from there)
SPLITS_DIR    = Path(ML_RESEARCH) / 'data' / 'splits'     # Where train.csv and the output live
AUGMENTED_DIR = Path(ML_RESEARCH) / 'data' / 'augmented'  # Where synthetic CSVs from LLMs are stored

# Aspect columns — must match model config exactly
ASPECT_COLUMNS = ['stayingpower', 'texture', 'smell', 'price', 'colour', 'shipping', 'packing']
LABELS         = ['positive', 'neutral', 'negative']

## Distribution Helpers

These functions count how many samples exist per class per aspect, and build the Markdown impact report.

In [ ]:
def get_class_counts(df: pd.DataFrame) -> dict:
    """Return {aspect: {label: count}} for all aspects."""
    dist = {}
    for aspect in ASPECT_COLUMNS:
        if aspect not in df.columns:
            continue
        vc = df[aspect].value_counts(dropna=True)   # Count each sentiment label
        dist[aspect] = {str(lbl): int(vc.get(lbl, 0)) for lbl in LABELS}
    return dist


def build_report(before: dict, after: dict, n_original: int, n_synthetic: int,
                 n_combined: int, synthetic_files: list) -> str:
    """Generate the augmentation_impact.md content as a string."""
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    # (Full implementation in the .py file — see create_train_aug.py)
    # This builds tables showing before/after class counts for each aspect
    # and a summary of which rare classes benefited most from augmentation.
    pass  # Full body in the .py source file

## Main Pipeline

In [ ]:
def main():
    log.info('=' * 65)
    log.info('Creating Augmented Training Set')
    log.info('=' * 65)

    # STEP 1: Load the clean original training data
    train_path = SPLITS_DIR / 'train.csv'
    if not train_path.exists():
        log.error('Original training file not found. Run preprocess_and_split.py first!')
        return
    train_df    = pd.read_csv(train_path)
    before_dist = get_class_counts(train_df)  # Snapshot class distribution BEFORE augmentation

    # STEP 2: Load all synthetic CSVs produced by convert_synthetic_robust.py
    csv_files = sorted(AUGMENTED_DIR.glob('*.csv'))
    if not csv_files:
        log.warning('No augmented CSV files found — nothing to merge.')
        return

    aug_dfs = []
    synthetic_file_info = []
    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        aug_dfs.append(df)
        synthetic_file_info.append((csv_path.name, len(df)))

    # STEP 3: Concatenate, then deduplicate
    synthetic_df = pd.concat(aug_dfs, ignore_index=True)

    # Remove exact text duplicates within the synthetic data (LLMs repeat themselves)
    synthetic_df = synthetic_df.drop_duplicates(subset=['data'])

    # Remove synthetic rows that appear verbatim in the real training set (data leakage prevention)
    synthetic_df = synthetic_df[~synthetic_df['data'].isin(train_df['data'])]

    # STEP 4: Combine — only keep columns present in BOTH DataFrames
    common_cols = [c for c in train_df.columns if c in synthetic_df.columns]
    combined_df = pd.concat([train_df[common_cols], synthetic_df[common_cols]], ignore_index=True)

    # STEP 5: Save the augmented training CSV
    out_path = SPLITS_DIR / 'train_augmented.csv'
    combined_df.to_csv(out_path, index=False)
    log.info(f'Saved augmented training set → {out_path}')

    # STEP 6: Generate and save the impact report
    after_dist  = get_class_counts(combined_df)
    report      = build_report(before_dist, after_dist, len(train_df),
                               len(synthetic_df), len(combined_df), synthetic_file_info)
    Path('augmentation_impact.md').write_text(report, encoding='utf-8')


main()